# Contracting Lorenz Map: Invariant Density Computation

This notebook demonstrates the computation of the invariant density for the contracting Lorenz map using the rigorous Ulam method.

## System Description

The contracting Lorenz map is a Poincaré section of the Lorenz attractor with:
- Strong contraction (ρ < 1)
- Nonlinear dynamics (power laws x^s, x^r)
- Singularity at x=0
- Additive noise for regularization

In [ ]:
# Activate notebook environment
import Pkg
Pkg.activate(@__DIR__)

using RigorousUlam2D
using IntervalArithmetic, IntervalBoxes
using LinearAlgebra, SparseArrays
using CairoMakie
using Printf

## Define the Contracting Lorenz Map

The map has a discontinuity at x=0, which requires special handling for interval arithmetic.

In [ ]:
function contracting_lorenz(x, y; s=2.5, r=6.0, ρ=0.7, c0=-0.25, c1=0.25)
    ax = abs(x)
    
    # For interval arguments, check which branch we're on
    if x isa Interval
        if inf(x) > 0
            # Entirely in positive branch (x > 0)
            return (-ρ * ax^s + 0.5, y * ax^r + c0)
        elseif sup(x) < 0
            # Entirely in negative branch (x < 0)
            return (ρ * ax^s - 0.5, -y * ax^r + c1)
        else
            # Contains x=0: take conservative hull of both branches
            Tx_pos = -ρ * ax^s + 0.5
            Gxy_pos = y * ax^r + c0
            Tx_neg = ρ * ax^s - 0.5
            Gxy_neg = -y * ax^r + c1
            return (RigorousUlam2D.hull(Tx_pos, Tx_neg), RigorousUlam2D.hull(Gxy_pos, Gxy_neg))
        end
    else
        # For dual numbers (ForwardDiff)
        x_val = x isa Real ? x : ForwardDiff.value(x)
        if x_val > 0
            return (-ρ * ax^s + 0.5, y * ax^r + c0)
        else
            return (ρ * ax^s - 0.5, -y * ax^r + c1)
        end
    end
end

# System parameters
domain = IntervalBox(interval(-0.5, 0.5), interval(-0.5, 0.5))
noise = uniform_noise_box(0.01, 2)

println("System: Contracting Lorenz map")
println("Domain: [-0.5, 0.5]²")
println("Noise: ±0.01 uniform")

## Build Adaptive Partition

We use adaptive partitioning to efficiently handle the contracting dynamics.

In [ ]:
target_diameter = 0.03

println("Building adaptive partition...")
@time partition, graph = build_adaptive_partition(
    contracting_lorenz, domain, noise,
    max_diameter=target_diameter,
    initial_depth=2,
    safety_margin=1.1,
    enclosure_method=zeroth_order_enclosure,
)

println("\nPartition statistics:")
println("  Boxes: ", partition.n_boxes)
println("  Graph non-zeros: ", nnz(graph))
@printf("  Graph sparsity: %.1f%%\n", 100*(1 - nnz(graph)/partition.n_boxes^2))

## Build Transition Matrix with Noise

Using operator composition: P_noisy = N * P

In [ ]:
println("Building deterministic transition matrix...")
@time P_det, U = build_transition_matrix(
    contracting_lorenz, partition,
    transition_graph=graph,
    enclosure_method=zeroth_order_enclosure,
    maxdepth=10,
    ε=1e-5
)

println("\nBuilding noise convolution operator...")
@time N = build_noise_convolution_operator(
    noise, partition,
    transition_graph=graph,
    ε=1e-5,
    max_iterations=20
)

println("\nComposing operators: P_noisy = N * P_det")
P_noisy = N * P_det

println("\nMatrix statistics:")
@printf("  Size: %d × %d\n", size(P_noisy)...)
@printf("  Non-zeros: %d\n", nnz(P_noisy))
@printf("  Sparsity: %.1f%%\n", 100*(1 - nnz(P_noisy)/prod(size(P_noisy))))

## Compute Invariant Measure

Find the leading eigenvector of the normalized transition matrix.

In [ ]:
# Normalize to get stochastic matrix
println("Normalizing transition matrix...")
col_sums = normalize_columns!(P_noisy)
@printf("Column sum range: [%.6f, %.6f]\n", minimum(col_sums), maximum(col_sums))

# Compute invariant measure via eigendecomposition
println("\nComputing invariant measure (leading eigenvector)...")
@time begin
    λ, v = eigen(Matrix(P_noisy))
    
    # Find eigenvalue closest to 1
    idx = argmin(abs.(λ .- 1.0))
    
    # Extract and normalize invariant measure
    μ = real(v[:, idx])
    μ ./= sum(μ)  # Normalize to probability measure
end

println("\nInvariant measure statistics:")
@printf("  Leading eigenvalue: %.10f\n", abs(λ[idx]))
@printf("  Max density: %.6f\n", maximum(μ))
@printf("  Active boxes (μ > 1e-4): %d / %d\n", count(μ .> 1e-4), partition.n_boxes)

## Visualize Invariant Density

Plot the invariant density as a heatmap over the partition boxes.

In [ ]:
# Prepare data for plotting
function prepare_density_data(partition, measure)
    # Create arrays for box centers and densities
    x_centers = Float64[]
    y_centers = Float64[]
    densities = Float64[]
    
    for (i, box) in enumerate(partition.boxes)
        # Compute box center
        x_center = mid(box[1])
        y_center = mid(box[2])
        
        # Density = measure / box_area
        density = measure[i] / box_measure(box)
        
        push!(x_centers, x_center)
        push!(y_centers, y_center)
        push!(densities, density)
    end
    
    return x_centers, y_centers, densities
end

x_centers, y_centers, densities = prepare_density_data(partition, μ)

# Create the plot
fig = Figure(size=(800, 700))
ax = Axis(fig[1, 1],
    xlabel="x",
    ylabel="y",
    title="Invariant Density - Contracting Lorenz Map",
    aspect=1
)

# Plot density as scatter with color
sc = scatter!(ax, x_centers, y_centers,
    color=densities,
    colormap=:viridis,
    markersize=8,
    colorrange=(0, maximum(densities))
)

# Add colorbar
Colorbar(fig[1, 2], sc, label="Density ρ(x,y)")

# Add domain boundary
lines!(ax, [-0.5, 0.5, 0.5, -0.5, -0.5], [-0.5, -0.5, 0.5, 0.5, -0.5],
    color=:white, linestyle=:dash, linewidth=2)

fig

## Additional Analysis: Spectral Properties

In [ ]:
# Sort eigenvalues by magnitude
λ_sorted = sort(abs.(λ), rev=true)

println("Leading eigenvalues:")
for i in 1:min(10, length(λ_sorted))
    @printf("  λ_%d = %.8f\n", i, λ_sorted[i])
end

if length(λ_sorted) >= 2
    gap = 1.0 - λ_sorted[2]
    rate = -log(λ_sorted[2])
    @printf("\nSpectral properties:\n")
    @printf("  Spectral gap: %.6f\n", gap)
    @printf("  Mixing rate: %.6f\n", rate)
    @printf("  Mixing time: ~%.1f iterations\n", 1/rate)
end

## Summary

This notebook demonstrated:
1. Adaptive partitioning for a contracting map with singularities
2. Operator composition architecture: P_noisy = N * P
3. Rigorous computation of the invariant density
4. Visualization of the resulting density

The invariant density shows concentration near the attractor region, with the noise providing regularization.